# ACE 1O86 redocking validation

1O86 is human testis ACE co-crystallized with lisinopril (`LPR`). The active-site Zn²⁺ is retained in the receptor; the crystallographic LPR centroid defines the docking box. Run this before candidate docking.

**Setup notes (tested with Meeko 0.8.0 and Vina 1.2.5):**
- The receptor is read with `--read_pdb` (no ProDy needed); Meeko adds the polar hydrogens and types the Zn.
- The ligand and docking box are defined **first**, because crystal structures often contain residues with unresolved side-chain atoms that Meeko cannot parameterize. Such residues are deleted only if they lie far outside the box (`BAD_RESIDUE_MIN_DISTANCE_FROM_BOX_A`); an incomplete residue near the pocket stops the run instead of being silently removed. The full Meeko log and a table of incomplete residues are saved.
- After preparation the notebook verifies that one Zn is present and that every residue within `POCKET_CHECK_RADIUS_A` of lisinopril survived.
- Redocking passes when at least `MIN_PASSING_SEEDS` seeds reproduce the crystal pose within 2 Å.

In [1]:
# Colab-only dependency check. Nothing runs on VS Code, or when the tools are already present (see notebook 0 / requirements.txt).
import importlib.util, shutil, subprocess, sys
if importlib.util.find_spec('google.colab'):
    if shutil.which('mk_prepare_receptor.py') is None or importlib.util.find_spec('gemmi') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'meeko==0.8.0', 'gemmi', 'rdkit', 'biopython'])  # ProDy is not needed (--read_pdb)
    if shutil.which('vina') is None:
        subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'autodock-vina'])  # Colab (Ubuntu 24.04) ships Vina 1.2.5


In [2]:
# Colab setup, when required: %pip install meeko rdkit biopython pandas
# Also install an AutoDock Vina executable and make `vina` available on PATH.
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/ACE_Workflow')
except ImportError:
    ROOT = Path(os.environ.get('ACE_WORKFLOW_ROOT', Path.cwd())).resolve()
    if not (ROOT / 'ace_config.py').exists(): ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ace_config.py').exists()), ROOT)
if not (ROOT / 'ace_config.py').exists(): raise FileNotFoundError('Open the Google Drive-synced ACE_Workflow folder in VS Code.')
os.environ['ACE_WORKFLOW_ROOT'] = str(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from ace_config import *

import json, re, shutil, subprocess
import numpy as np, pandas as pd
from Bio.PDB import PDBIO, PDBParser, Select
from rdkit import Chem
from rdkit.Chem import AllChem

RAW_PDB = INPUTS['RECEPTOR_1O86']
assert RAW_PDB.exists(), f'Missing 1O86 receptor: {RAW_PDB}'
VINA = shutil.which('vina') or shutil.which('vina.exe')
MEEKO_RECEPTOR = shutil.which('mk_prepare_receptor.py')
MEEKO_LIGAND = shutil.which('mk_prepare_ligand.py')
if not all([VINA, MEEKO_RECEPTOR, MEEKO_LIGAND]):
    raise RuntimeError('Install AutoDock Vina and Meeko, then ensure vina/mk_prepare_* are on PATH.')
SEEDS, EXHAUSTIVENESS, NUM_MODES, BOX_PADDING_A = [7, 42, 2026], 16, 20, 8.0
BAD_RESIDUE_MIN_DISTANCE_FROM_BOX_A = 6.0  # incomplete residues are deleted ONLY if their centre is this far outside the docking box
POCKET_CHECK_RADIUS_A = 8.0                # after preparation, every residue within this distance of lisinopril must still be present
MIN_PASSING_SEEDS = 2  # a single lucky seed is not evidence that the setup reproduces the crystal pose
_v = subprocess.run([VINA, '--version'], text=True, capture_output=True)
VINA_VERSION = ((_v.stdout or _v.stderr).strip().splitlines() or ['unknown'])[0]
print('Vina:', VINA_VERSION)
if re.search(r'\b1\.1\.', VINA_VERSION): print('WARNING: Vina 1.1.x detected; use Vina >= 1.2 (see requirements/README).')
DOCKING_CPU = CPU_WORKERS  # CPU-only Vina run; no GPU is requested

Mounted at /content/drive
Vina: AutoDock Vina v1.2.5


In [3]:
# Parse 1O86, extract the crystallographic LPR pose, assign its known lisinopril bond orders, and define the docking box.
# (Done before receptor preparation because the box decides which incomplete residues may safely be deleted.)
parser = PDBParser(QUIET=True)
structure = parser.get_structure('1O86', str(RAW_PDB))
class LPRSelector(Select):
    def accept_residue(self, residue): return residue.get_resname().strip() == 'LPR' and residue.get_parent().id == 'A'
    def accept_atom(self, atom): return atom.get_altloc() in (' ', 'A')
crystal_pdb = PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal.pdb'
io = PDBIO(); io.set_structure(structure); io.save(str(crystal_pdb), LPRSelector())
# Lisinopril (C21H31N3O5, 29 heavy atoms). The previous template was a different molecule (C22H34N4O4) and failed to match.
template = Chem.MolFromSmiles('NCCCC[C@H](N[C@@H](CCc1ccccc1)C(=O)O)C(=O)N1CCC[C@H]1C(=O)O')
crystal_raw = Chem.MolFromPDBFile(str(crystal_pdb), removeHs=False, sanitize=False)
if crystal_raw is None or crystal_raw.GetNumAtoms() != template.GetNumHeavyAtoms():
    raise ValueError(f'LPR in chain A has {0 if crystal_raw is None else crystal_raw.GetNumAtoms()} atoms; lisinopril has {template.GetNumHeavyAtoms()}.')
crystal_mol = AllChem.AssignBondOrdersFromTemplate(template, crystal_raw)
Chem.SanitizeMol(crystal_mol)
crystal_sdf = PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal.sdf'
# Meeko needs explicit hydrogens; add them with coordinates (heavy-atom positions stay at the crystal pose).
crystal_h = Chem.AddHs(crystal_mol, addCoords=True)
writer = Chem.SDWriter(str(crystal_sdf)); writer.write(crystal_h); writer.close()
ligand_pdbqt = PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal.pdbqt'
run = subprocess.run([MEEKO_LIGAND, '-i', str(crystal_sdf), '-o', str(ligand_pdbqt)], text=True, capture_output=True)
if run.returncode: raise RuntimeError(run.stderr[-3000:])
coords = np.array([list(crystal_mol.GetConformer().GetAtomPosition(i)) for i in range(crystal_mol.GetNumAtoms()) if crystal_mol.GetAtomWithIdx(i).GetAtomicNum() > 1])
lig_heavy_xyz = coords
center, span = coords.mean(axis=0), np.ptp(coords, axis=0)  # ndarray.ptp was removed in NumPy 2
box = {'center_x': float(center[0]), 'center_y': float(center[1]), 'center_z': float(center[2]),
       'size_x': float(max(18.0, span[0] + BOX_PADDING_A)), 'size_y': float(max(18.0, span[1] + BOX_PADDING_A)), 'size_z': float(max(18.0, span[2] + BOX_PADDING_A)),
       'source': '1O86 chain A LPR (lisinopril) heavy-atom centroid'}
box_file = PATHS['REDOCKING_ANALYSIS'] / '1o86_docking_box.json'
box_file.write_text(json.dumps(box, indent=2))
box

[03:34:14] WARNING: More than one matching pattern found - picking one



{'center_x': 41.01362068965516,
 'center_y': 34.325206896551734,
 'center_z': 46.44120689655172,
 'size_x': 18.0,
 'size_y': 18.048000000000002,
 'size_z': 19.865000000000002,
 'source': '1O86 chain A LPR (lisinopril) heavy-atom centroid'}

In [4]:
# Receptor preparation: audit incomplete residues, then let Meeko delete only residues that are far from the docking box.
from scipy.spatial.distance import cdist

# Full heavy-atom counts per standard residue (no OXT, no hydrogens).
EXPECTED_HEAVY = {'ALA': 5, 'ARG': 11, 'ASN': 8, 'ASP': 8, 'CYS': 6, 'GLN': 9, 'GLU': 9, 'GLY': 4, 'HIS': 10, 'ILE': 8,
                  'LEU': 8, 'LYS': 9, 'MET': 8, 'PHE': 11, 'PRO': 7, 'SER': 6, 'THR': 7, 'TRP': 14, 'TYR': 12, 'VAL': 7}
audit = []
for residue in structure[0]['A']:
    name = residue.get_resname().strip()
    if residue.id[0] != ' ' or name not in EXPECTED_HEAVY: continue
    present = {a.get_id() for a in residue if a.get_id() != 'OXT' and (a.element or '').upper() != 'H'}
    missing = EXPECTED_HEAVY[name] - len(present)
    if missing > 0:
        xyz = np.array([a.coord for a in residue])
        audit.append({'residue': f'A:{residue.id[1]}', 'resname': name, 'missing_heavy_atoms': missing,
                      'min_dist_to_LPR_A': round(float(cdist(xyz, lig_heavy_xyz).min()), 1)})
audit = pd.DataFrame(audit)
audit.to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_incomplete_residues.csv', index=False)
print(f'{len(audit)} chain-A residues have missing heavy atoms (unresolved in the crystal).')
if len(audit): display(audit.sort_values('min_dist_to_LPR_A'))

class ACEReceptorSelector(Select):
    def accept_chain(self, chain): return chain.id == 'A'
    def accept_residue(self, residue):
        # Bio.PDB residue.id is (hetflag, resseq, icode); the residue NAME is residue.get_resname().
        return residue.id[0] == ' ' or residue.get_resname().strip() == 'ZN'
    def accept_atom(self, atom): return atom.get_altloc() in (' ', 'A')  # one alternate location only

receptor_pdb = PATHS['REDOCKING_RECEPTOR'] / '1o86_chainA_with_Zn.pdb'
io = PDBIO(); io.set_structure(structure); io.save(str(receptor_pdb), ACEReceptorSelector())
# Blank the alt-loc column so Meeko does not need --default_altloc.
receptor_lines = [(l[:16] + ' ' + l[17:]) if l.startswith(('ATOM', 'HETATM')) else l for l in receptor_pdb.read_text().splitlines()]
receptor_pdb.write_text('\n'.join(receptor_lines) + '\n')

# --- Zn coordination audit and histidine tautomers -------------------------------------------------------------
# Meeko protonates HIS as HIE (hydrogen on NE2) by default. A histidine that binds the Zn through NE2 must be HID
# (hydrogen on ND1); otherwise a hydrogen is placed on the coordinating nitrogen, about 1 A from the metal.
zn_residues = [r for r in structure[0]['A'] if r.get_resname().strip() == 'ZN']
if len(zn_residues) != 1: raise RuntimeError(f'Expected one Zn in chain A, found {len(zn_residues)}.')
zn_xyz = np.array(list(zn_residues[0])[0].coord)
zn_rows = []
for residue in structure[0]['A']:
    if residue.id[0] != ' ': continue
    for atom in residue:
        element = (atom.element or atom.get_id()[:1]).upper()
        d = float(np.linalg.norm(atom.coord - zn_xyz))
        if element in {'N', 'O', 'S'} and d <= 2.8:
            zn_rows.append({'residue_number': residue.id[1], 'resname': residue.get_resname().strip(), 'atom': atom.get_id(), 'distance_A': round(d, 2)})
zn_contacts = pd.DataFrame(zn_rows, columns=['residue_number', 'resname', 'atom', 'distance_A']).sort_values('distance_A')
zn_contacts.to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_zn_coordination.csv', index=False)
print('Protein atoms within 2.8 A of the catalytic Zn:'); display(zn_contacts)
if len(zn_contacts) != 3: print(f'WARNING: {len(zn_contacts)} protein ligands found; the ACE Zn site is normally coordinated by 2 His + 1 Glu. Inspect the structure.')
hid = sorted(zn_contacts.query("resname == 'HIS' and atom == 'NE2'").residue_number.unique())   # NE2 binds Zn -> H on ND1
hie = sorted(zn_contacts.query("resname == 'HIS' and atom == 'ND1'").residue_number.unique())   # ND1 binds Zn -> H on NE2
set_template = ','.join([f'A:{n}=HID' for n in hid] + [f'A:{n}=HIE' for n in hie])
print('Histidine tautomers passed to Meeko:', set_template or '(none)')

output_base = PATHS['REDOCKING_RECEPTOR'] / '1o86_receptor'
meeko_log = PATHS['REDOCKING_RECEPTOR'] / '1o86_receptor_meeko.log'
# --read_pdb avoids ProDy; Meeko adds polar hydrogens and types the Zn.
# --delete_bad_res_from_box_radius: unmatched (incomplete) residues are deleted only when their centre is more than
# BAD_RESIDUE_MIN_DISTANCE_FROM_BOX_A outside the box; any unmatched residue nearer the pocket makes Meeko stop with an error.
cmd = [MEEKO_RECEPTOR, '--read_pdb', str(receptor_pdb), '-o', str(output_base), '--write_pdbqt',
       '--box_center', *[f'{box[k]:.3f}' for k in ('center_x', 'center_y', 'center_z')],
       '--box_size', *[f'{box[k]:.3f}' for k in ('size_x', 'size_y', 'size_z')],
       '--delete_bad_res_from_box_radius', str(BAD_RESIDUE_MIN_DISTANCE_FROM_BOX_A)]
if set_template: cmd += ['--set_template', set_template]
run = subprocess.run(cmd, text=True, capture_output=True)
log_text = run.stdout + run.stderr
meeko_log.write_text(log_text)                       # full log; never truncated
unmatched = sorted(set(re.findall(r"No template matched for residue_key='([^']+)'", log_text)))
deleted = re.findall(r'Ignored (\S+): ([\d.]+) A outside box', log_text)
receptor_pdbqt = output_base.with_suffix('.pdbqt')
if run.returncode or not receptor_pdbqt.exists():
    near = [r for r in unmatched if r not in {d[0] for d in deleted}]
    raise RuntimeError(
        f'Meeko could not prepare the receptor. Unmatched residues: {unmatched}. Not deletable because they lie within '
        f'{BAD_RESIDUE_MIN_DISTANCE_FROM_BOX_A} A of the docking box (pocket region): {near}. '
        f'Repair them (e.g. PDBFixer / Modeller) or choose a different structure; do not delete pocket residues blindly. '
        f'Full log: {meeko_log}\n--- last lines ---\n' + log_text[-1200:])
print(f'Unmatched residues: {unmatched}')
print('Deleted (outside box, not part of the pocket):', [f'{r} ({d} A outside)' for r, d in deleted])

# Sanity checks: exactly one catalytic Zn, polar hydrogens present, and no pocket residue lost.
pdbqt_atoms = [l for l in receptor_pdbqt.read_text().splitlines() if l.startswith(('ATOM', 'HETATM'))]
atom_types = [l[77:79].strip() for l in pdbqt_atoms]
n_zn, n_hd = atom_types.count('Zn'), atom_types.count('HD')
if n_zn != 1: raise RuntimeError(f'Expected one Zn in the receptor PDBQT, found {n_zn}.')
if n_hd == 0: raise RuntimeError('Receptor PDBQT has no polar hydrogens (HD); check Meeko output.')
# No polar hydrogen may sit on a Zn-coordinating atom (would be < ~2.5 A from the metal).
zn_pdbqt = [np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])]) for l in pdbqt_atoms if l[77:79].strip() == 'Zn']
h_near_zn = [(l[17:20].strip(), l[22:26].strip(), l[12:16].strip()) for l in pdbqt_atoms if l[77:79].strip() == 'HD'
             and np.linalg.norm(np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])]) - zn_pdbqt[0]) < 2.5]
if h_near_zn: raise RuntimeError(f'Polar hydrogen(s) within 2.5 A of the Zn in the prepared receptor: {h_near_zn}. Fix the coordinating residue protonation (--set_template).')
kept = {(l[21], int(l[22:26])) for l in pdbqt_atoms}
pocket = set()
for residue in structure[0]['A']:
    if residue.id[0] == ' ':
        xyz = np.array([a.coord for a in residue])
        if cdist(xyz, lig_heavy_xyz).min() <= POCKET_CHECK_RADIUS_A: pocket.add(('A', residue.id[1]))
lost = sorted(pocket - kept)
if lost: raise RuntimeError(f'Pocket residues (within {POCKET_CHECK_RADIUS_A} A of lisinopril) missing from the prepared receptor: {lost}')
print(f'Receptor prepared: {receptor_pdbqt} ({len(pdbqt_atoms)} atoms, {n_hd} polar H, {n_zn} Zn); '
      f'all {len(pocket)} residues within {POCKET_CHECK_RADIUS_A} A of lisinopril are present.')


9 chain-A residues have missing heavy atoms (unresolved in the crystal).


,residue,resname,missing_heavy_atoms,min_dist_to_LPR_A
1,A:64,GLU,4,12.7
4,A:303,GLU,4,17.1
5,A:307,LYS,4,20.1
2,A:117,LYS,4,21.3
3,A:298,SER,1,21.6
7,A:440,ASP,3,27.1
6,A:439,SER,2,27.8
8,A:618,GLN,8,28.3
0,A:40,ASP,3,34.0


Protein atoms within 2.8 A of the catalytic Zn:


,residue_number,resname,atom,distance_A
2,411,GLU,OE1,2.00
0,383,HIS,NE2,2.04
1,387,HIS,NE2,2.07


Histidine tautomers passed to Meeko: A:383=HID,A:387=HID
Unmatched residues: ['A:117', 'A:298', 'A:303', 'A:307', 'A:40', 'A:439', 'A:440', 'A:618', 'A:64']
Deleted (outside box, not part of the pocket): ['A:40 (27.3462 A outside)', 'A:64 (9.1625 A outside)', 'A:117 (14.5680 A outside)', 'A:298 (11.1683 A outside)', 'A:303 (9.5205 A outside)', 'A:307 (13.2833 A outside)', 'A:439 (18.9908 A outside)', 'A:440 (18.6963 A outside)', 'A:618 (20.6314 A outside)']
Receptor prepared: /content/drive/MyDrive/ACE_Workflow/redocking_1o86/receptor/1o86_receptor.pdbqt (5610 atoms, 989 polar H, 1 Zn); all 55 residues within 8.0 A of lisinopril are present.


In [5]:
runs = []
for seed in SEEDS:
    pose = PATHS['REDOCKING_DOCKING'] / f'LPR_redocked_seed_{seed}.pdbqt'
    log = PATHS['REDOCKING_DOCKING'] / f'LPR_redocked_seed_{seed}.log'
    cmd = [VINA, '--receptor', str(receptor_pdbqt), '--ligand', str(ligand_pdbqt),
           '--center_x', str(box['center_x']), '--center_y', str(box['center_y']), '--center_z', str(box['center_z']),
           '--size_x', str(box['size_x']), '--size_y', str(box['size_y']), '--size_z', str(box['size_z']),
           '--cpu', str(DOCKING_CPU), '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(NUM_MODES), '--seed', str(seed), '--out', str(pose)]  # Vina >= 1.2 has no --log option
    result = subprocess.run(cmd, text=True, capture_output=True)
    log.write_text(result.stdout + result.stderr)  # keep the Vina console output as the run log
    if result.returncode: raise RuntimeError(f'Seed {seed} failed: {result.stderr[-2000:]}')
    affinity = next((float(line.split()[3]) for line in pose.read_text(errors='replace').splitlines() if line.startswith('REMARK VINA RESULT')), np.nan)
    runs.append({'seed': seed, 'pose_pdbqt': str(pose), 'affinity_kcal_mol': affinity})
redock = pd.DataFrame(runs); display(redock)

,seed,pose_pdbqt,affinity_kcal_mol
0,7,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.889
1,42,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.979
2,2026,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.912


In [6]:
# Direct RMSD in the receptor frame; coordinates are not translated or superposed.
def pdbqt_heavy_coordinates(path):
    values, started = [], False
    for line in Path(path).read_text(errors='replace').splitlines():
        if line.startswith('MODEL'): started = True; continue
        if started and line.startswith('ENDMDL'): break
        if line.startswith(('ATOM', 'HETATM')) and line[77:79].strip().upper() not in {'H', 'HD', 'HS'}:
            values.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.asarray(values)
reference = pdbqt_heavy_coordinates(ligand_pdbqt)
def direct_rmsd(a, b): return float(np.sqrt(np.mean(np.sum((a - b) ** 2, axis=1))))
redock['heavy_atom_rmsd_A'] = redock.pose_pdbqt.map(lambda p: direct_rmsd(reference, pdbqt_heavy_coordinates(p)) if len(pdbqt_heavy_coordinates(p)) == len(reference) else np.nan)
redock['pass_2A'] = redock.heavy_atom_rmsd_A <= 2.0
n_pass = int(redock.pass_2A.sum()); redocking_passed = n_pass >= MIN_PASSING_SEEDS
best = redock.sort_values(['heavy_atom_rmsd_A', 'affinity_kcal_mol']).iloc[0]
best_pose = PATHS['REDOCKING_DOCKING'] / 'LPR_redocked_best.pdbqt'
shutil.copy2(best.pose_pdbqt, best_pose)
redock.to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_summary.csv', index=False)
(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_metadata.json').write_text(json.dumps({'raw_pdb': str(RAW_PDB), 'raw_pdb_sha256': sha256_file(RAW_PDB), 'receptor_pdbqt': str(receptor_pdbqt), 'box': box, 'vina_version': VINA_VERSION, 'seeds': SEEDS, 'n_seeds_pass_2A': n_pass, 'min_passing_seeds': MIN_PASSING_SEEDS, 'redocking_passed': redocking_passed, 'median_rmsd_A': float(redock.heavy_atom_rmsd_A.median()), 'best_seed': int(best.seed), 'best_rmsd_A': float(best.heavy_atom_rmsd_A)}, indent=2))
display(redock.sort_values('heavy_atom_rmsd_A'))
print(f'{n_pass}/{len(redock)} seeds within 2 Å (median RMSD {redock.heavy_atom_rmsd_A.median():.2f} Å). ' + ('Redocking PASSED.' if redocking_passed else 'Redocking FAILED: resolve the setup before candidate docking.'))

,seed,pose_pdbqt,affinity_kcal_mol,heavy_atom_rmsd_A,pass_2A
0,7,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.889,9.432020,False
1,42,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.979,9.460319,False
2,2026,/content/drive/MyDrive/ACE_Workflow/redocking_...,-7.912,9.484436,False


0/3 seeds within 2 Å (median RMSD 9.46 Å). Redocking FAILED: resolve the setup before candidate docking.


## Redocking diagnostics (runs only if the gate above fails)

A reproducible wrong pose (all seeds at the same RMSD) points at the **scoring function or the ligand/receptor model**, not at random search failure. This section separates the possible causes:

1. **Does Vina score the crystal pose better than the pose it found?** If the crystal pose scores worse, Vina prefers the wrong pose and more sampling will not help.
2. **Does the crystal pose survive local minimisation?** A large drift means the prepared receptor/ligand (hydrogens, Zn typing) is inconsistent with the crystal geometry.
3. **Is the Zn site occupied?** Distance from the Zn to the closest ligand O/N/S in the crystal pose and in the best docked pose.
4. **Ionised ligand variant.** Lisinopril is a zwitterion at physiological pH (two carboxylates, protonated lysine amine); the neutral form used above lacks the Zn-binding carboxylate.

In [7]:
# Redocking diagnostics. Set DIAGNOSE_ALWAYS = True to run them even when redocking passed.
DIAGNOSE_ALWAYS = False
RUN_IONIZED_VARIANT = True        # extra Vina runs (a few minutes each on CPU)
VARIANT_SEEDS = SEEDS[:2]

if redocking_passed and not DIAGNOSE_ALWAYS:
    print('Redocking passed; diagnostics skipped.')
else:
    ELEM = {'A': 'C', 'NA': 'N', 'OA': 'O', 'SA': 'S'}
    def pdbqt_atoms(path, heavy_only=True):
        out, started = [], False
        for line in Path(path).read_text(errors='replace').splitlines():
            if line.startswith('MODEL'): started = True; continue
            if started and line.startswith('ENDMDL'): break
            if line.startswith(('ATOM', 'HETATM')):
                t = line[77:79].strip()
                if heavy_only and t.upper() in {'H', 'HD', 'HS'}: continue
                out.append((ELEM.get(t, t[:1].upper()), np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])))
        return out
    box_args = ['--center_x', str(box['center_x']), '--center_y', str(box['center_y']), '--center_z', str(box['center_z']),
                '--size_x', str(box['size_x']), '--size_y', str(box['size_y']), '--size_z', str(box['size_z'])]
    def vina_run(ligand, extra):
        r = subprocess.run([VINA, '--receptor', str(receptor_pdbqt), '--ligand', str(ligand), *box_args, '--cpu', str(DOCKING_CPU), *extra], text=True, capture_output=True)
        if r.returncode: raise RuntimeError(r.stderr[-1500:] or r.stdout[-1500:])
        return r.stdout
    def pose_score(path):
        return next((float(l.split()[3]) for l in Path(path).read_text(errors='replace').splitlines() if l.startswith('REMARK VINA RESULT')), np.nan)
    # pdbqt_atoms maps unknown types by their first letter, so the Zn atom type appears as 'Z'.
    zn_atoms = [xyz for t, xyz in pdbqt_atoms(receptor_pdbqt, heavy_only=False) if t == 'Z']
    if len(zn_atoms) != 1: raise RuntimeError(f'Expected one Zn in {receptor_pdbqt}, found {len(zn_atoms)}')
    zn = zn_atoms[0]
    def zn_contact(atoms):
        d = [(float(np.linalg.norm(xyz - zn)), el, i) for i, (el, xyz) in enumerate(atoms) if el in {'O', 'N', 'S'}]
        return min(d) if d else (np.nan, '', -1)

    top = redock.sort_values('affinity_kcal_mol').iloc[0]             # best-scoring seed (not best RMSD)
    crystal_atoms, docked_atoms = pdbqt_atoms(ligand_pdbqt), pdbqt_atoms(top.pose_pdbqt)

    # 1-2. Score of the crystal pose, and its local minimisation.
    so = vina_run(ligand_pdbqt, ['--score_only'])
    crystal_score = float(re.search(r'Estimated Free Energy of Binding\s*:\s*(-?\d+\.?\d*)', so).group(1))
    local_pose = PATHS['REDOCKING_DOCKING'] / 'LPR_crystal_local_only.pdbqt'
    lo = vina_run(ligand_pdbqt, ['--local_only', '--out', str(local_pose)])
    local_atoms = pdbqt_atoms(local_pose)
    local_rmsd = direct_rmsd(np.array([x for _, x in crystal_atoms]), np.array([x for _, x in local_atoms]))
    local_score = float(re.findall(r'Estimated Free Energy of Binding\s*:\s*(-?\d+\.?\d*)', lo)[-1])  # Vina prints it to the console for --local_only

    # 3. Zn contacts.
    cz, dz = zn_contact(crystal_atoms), zn_contact(docked_atoms)
    summary = {'crystal_pose_score_only': crystal_score, 'crystal_local_only_score': local_score, 'crystal_local_only_rmsd_A': round(local_rmsd, 2),
               'best_docked_score': float(top.affinity_kcal_mol), 'best_docked_rmsd_A': round(float(top.heavy_atom_rmsd_A), 2),
               'crystal_closest_Zn_contact_A': (round(cz[0], 2), cz[1]), 'docked_closest_Zn_contact_A': (round(dz[0], 2), dz[1])}
    display(pd.Series(summary, name='value').to_frame())

    hints = []
    if crystal_score > top.affinity_kcal_mol + 0.5:
        hints.append(f'Vina scores the crystal pose ({crystal_score:.1f}) worse than the pose it found ({top.affinity_kcal_mol:.1f}): a SCORING problem, not a sampling problem.')
    else:
        hints.append('The crystal pose scores about as well as the docked pose: a sampling/search problem is possible (try higher exhaustiveness).')
    if local_rmsd > 2.5: hints.append(f'The crystal pose drifts {local_rmsd:.1f} A on local minimisation: the prepared receptor/ligand is inconsistent with the crystal geometry (hydrogens, protonation, Zn typing).')
    if cz[0] <= 2.8 and dz[0] > 3.2: hints.append(f'In the crystal pose a ligand {cz[1]} binds Zn at {cz[0]:.1f} A, but the docked pose does not (closest {dz[0]:.1f} A): the Zn-binding interaction is not being reproduced.')
    print('\n'.join('- ' + h for h in hints))

    per_atom = pd.DataFrame({'ligand_atom_index': range(len(crystal_atoms)), 'element': [e for e, _ in crystal_atoms],
        'deviation_A': [float(np.linalg.norm(c[1] - d[1])) for c, d in zip(crystal_atoms, docked_atoms)],
        'dist_to_Zn_crystal_A': [float(np.linalg.norm(x - zn)) for _, x in crystal_atoms],
        'dist_to_Zn_docked_A': [float(np.linalg.norm(x - zn)) for _, x in docked_atoms]}).round(2)
    per_atom.to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_per_atom_deviation.csv', index=False)
    display(per_atom)

    # 4. Ionised (physiological pH) lisinopril: both carboxylates deprotonated, terminal lysine amine protonated.
    variant_rows = []
    if RUN_IONIZED_VARIANT:
        m = Chem.RWMol(Chem.Mol(crystal_mol))
        for match in m.GetSubstructMatches(Chem.MolFromSmarts('[CX3](=O)[OX2H1]')):
            o = m.GetAtomWithIdx(match[2]); o.SetFormalCharge(-1); o.SetNumExplicitHs(0); o.SetNoImplicit(True)
        for match in m.GetSubstructMatches(Chem.MolFromSmarts('[NX3;H2][CH2]')):
            a = m.GetAtomWithIdx(match[0]); a.SetFormalCharge(1); a.SetNumExplicitHs(3); a.SetNoImplicit(True)
        ion = m.GetMol(); Chem.SanitizeMol(ion)
        charge = Chem.GetFormalCharge(ion); print(f'Ionised lisinopril net charge: {charge:+d} (expected -1)')
        ion_sdf = PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal_ionized.sdf'
        w = Chem.SDWriter(str(ion_sdf)); w.write(Chem.AddHs(ion, addCoords=True)); w.close()
        ion_pdbqt = PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal_ionized.pdbqt'
        r = subprocess.run([MEEKO_LIGAND, '-i', str(ion_sdf), '-o', str(ion_pdbqt)], text=True, capture_output=True)
        if r.returncode or not ion_pdbqt.exists(): raise RuntimeError((r.stdout + r.stderr)[-1500:])
        ion_ref = pdbqt_heavy_coordinates(ion_pdbqt)
        for seed in VARIANT_SEEDS:
            pose = PATHS['REDOCKING_DOCKING'] / f'LPR_ionized_redocked_seed_{seed}.pdbqt'
            out = vina_run(ion_pdbqt, ['--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(NUM_MODES), '--seed', str(seed), '--out', str(pose)])
            (PATHS['REDOCKING_DOCKING'] / f'LPR_ionized_redocked_seed_{seed}.log').write_text(out)
            arr = pdbqt_heavy_coordinates(pose)
            variant_rows.append({'ligand_form': 'ionised', 'seed': seed, 'affinity_kcal_mol': pose_score(pose),
                                 'heavy_atom_rmsd_A': direct_rmsd(ion_ref, arr) if len(arr) == len(ion_ref) else np.nan})
        variants = pd.DataFrame(variant_rows); variants['pass_2A'] = variants.heavy_atom_rmsd_A <= 2.0
        display(variants); variants.to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_ionized_variant.csv', index=False)
        if variants.pass_2A.sum() >= min(MIN_PASSING_SEEDS, len(variants)):
            print('The ionised ligand reproduces the crystal pose: dock candidates in their ionised (pH ~7.4) form, not as neutral SMILES.')
        else:
            print('The ionised form does not fix it either; see the hints above (Vina lacks an explicit Zn-coordination term).')
    (PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_diagnostics.json').write_text(json.dumps({**summary, 'hints': hints, 'ionized_variant': variant_rows}, indent=2, default=float))


,value
crystal_pose_score_only,-7.771
crystal_local_only_score,-8.076
crystal_local_only_rmsd_A,0.25
best_docked_score,-7.979
best_docked_rmsd_A,9.46
crystal_closest_Zn_contact_A,"(2.14, O)"
docked_closest_Zn_contact_A,"(5.67, N)"


- The crystal pose scores about as well as the docked pose: a sampling/search problem is possible (try higher exhaustiveness).
- In the crystal pose a ligand O binds Zn at 2.1 A, but the docked pose does not (closest 5.7 A): the Zn-binding interaction is not being reproduced.


,ligand_atom_index,element,deviation_A,dist_to_Zn_crystal_A,dist_to_Zn_docked_A
0,0,C,4.92,4.62,7.98
1,1,N,7.41,4.63,9.38
2,2,C,9.54,4.16,10.42
3,3,C,10.78,2.69,11.53
4,4,O,11.60,2.14,12.17
5,5,O,11.40,2.61,11.95
6,6,C,12.18,4.86,11.33
7,7,C,13.65,5.41,11.19
8,8,C,14.18,6.55,9.90
9,9,C,14.97,7.63,9.70


Ionised lisinopril net charge: -1 (expected -1)


,ligand_form,seed,affinity_kcal_mol,heavy_atom_rmsd_A,pass_2A
0,ionised,7,-8.262,7.279875,False
1,ionised,42,-8.224,7.326261,False


The ionised form does not fix it either; see the hints above (Vina lacks an explicit Zn-coordination term).


## Where does the search go wrong? (mode-by-mode analysis, uses poses already on disk)

Vina writes up to 20 poses per run. For every pose this table gives the score, the RMSD to the crystal pose and the closest ligand O/S atom to the catalytic Zn.
- A crystal-like pose present but ranked lower means a **ranking/scoring** problem; none present means the **search** did not reach it (or it lies outside Vina's default 3 kcal/mol output window).
- Every ACE inhibitor class needs a Zn-binding group (carboxylate, thiol, phosphinate...). The **Zn-contact-restrained** line picks the best-scoring pose that has an O/S within 2.8 Å of the Zn. This is a knowledge-based restraint, not part of Vina; if it is used for validation it must also be used, unchanged, for candidate docking, and it will discard compounds that cannot reach the Zn.

In [8]:
ELEMENTS = {'A': 'C', 'NA': 'N', 'OA': 'O', 'SA': 'S'}
ZN_CONTACT_A = 2.8

def pdbqt_models(path):
    """All output modes of a Vina PDBQT: list of (score, heavy-atom coordinates, element array)."""
    models, score, xyz, el, inside = [], np.nan, [], [], False
    for line in Path(path).read_text(errors='replace').splitlines():
        if line.startswith('MODEL'): inside, score, xyz, el = True, np.nan, [], []
        elif inside and line.startswith('REMARK VINA RESULT'): score = float(line.split()[3])
        elif inside and line.startswith(('ATOM', 'HETATM')):
            t = line[77:79].strip()
            if t.upper() in {'H', 'HD', 'HS'}: continue
            xyz.append([float(line[30:38]), float(line[38:46]), float(line[46:54])]); el.append(ELEMENTS.get(t, t[:1].upper()))
        elif inside and line.startswith('ENDMDL'): models.append((score, np.asarray(xyz), np.array(el))); inside = False
    return models

def receptor_zn_xyz(path):
    for line in Path(path).read_text().splitlines():
        if line.startswith(('ATOM', 'HETATM')) and line[77:79].strip() == 'Zn':
            return np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    raise RuntimeError(f'No Zn atom in {path}')
ZN_XYZ = receptor_zn_xyz(receptor_pdbqt)

def mode_table(pose_file, ref):
    rows = []
    for rank, (score, xyz, el) in enumerate(pdbqt_models(pose_file), 1):
        d = np.linalg.norm(xyz[np.isin(el, ['O', 'S'])] - ZN_XYZ, axis=1)
        rows.append({'mode': rank, 'score': score, 'rmsd_A': direct_rmsd(ref, xyz) if len(xyz) == len(ref) else np.nan,
                     'min_Zn_OS_dist_A': float(d.min()) if len(d) else np.nan})
    return pd.DataFrame(rows)

def summarize_modes(t):
    """One-row summary of a mode table (unrestrained top, best-RMSD mode, Zn-contact-restrained top)."""
    top, best = t.iloc[0], t.loc[t.rmsd_A.idxmin()]
    zn = t[t.min_Zn_OS_dist_A <= ZN_CONTACT_A]
    r = {'n_modes': len(t), 'top_score': top.score, 'top_rmsd_A': round(top.rmsd_A, 2), 'top_Zn_OS_dist_A': round(top.min_Zn_OS_dist_A, 2),
         'lowest_rmsd_A': round(best.rmsd_A, 2), 'lowest_rmsd_mode': int(best['mode']), 'lowest_rmsd_mode_score': best.score,
         'n_modes_with_Zn_contact': len(zn)}
    if len(zn):
        z = zn.iloc[0]; r.update({'zn_restrained_mode': int(z['mode']), 'zn_restrained_score': z.score, 'zn_restrained_rmsd_A': round(z.rmsd_A, 2)})
    else:
        r.update({'zn_restrained_mode': np.nan, 'zn_restrained_score': np.nan, 'zn_restrained_rmsd_A': np.nan})
    return r

FORMS = {'neutral': (PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal.pdbqt', 'LPR_redocked_seed_{seed}.pdbqt'),
         'ionised': (PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal_ionized.pdbqt', 'LPR_ionized_redocked_seed_{seed}.pdbqt')}
all_modes, summaries = [], []
for form, (ref_file, pattern) in FORMS.items():
    if not Path(ref_file).exists(): continue
    ref = pdbqt_heavy_coordinates(ref_file)
    for seed in SEEDS:
        pose_file = PATHS['REDOCKING_DOCKING'] / pattern.format(seed=seed)
        if not pose_file.exists(): continue
        t = mode_table(pose_file, ref)
        all_modes.append(t.assign(form=form, seed=seed)); summaries.append({'form': form, 'seed': seed, **summarize_modes(t)})
if not summaries:
    print('No pose files found; run the redocking cells first.')
else:
    pd.concat(all_modes).to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_mode_analysis.csv', index=False)
    mode_summary = pd.DataFrame(summaries); mode_summary.to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_mode_summary.csv', index=False)
    display(mode_summary.T)
    for r in mode_summary.itertuples():
        tag = f'{r.form}/seed {r.seed}'
        if r.lowest_rmsd_A <= 2.0 and r.lowest_rmsd_mode > 1:
            print(f'{tag}: a crystal-like pose exists at mode {r.lowest_rmsd_mode} ({r.lowest_rmsd_mode_score - r.top_score:+.2f} kcal/mol vs the top mode): a RANKING/SCORING problem.')
        elif r.lowest_rmsd_A <= 2.0: print(f'{tag}: the top mode is crystal-like.')
        else: print(f'{tag}: no mode within 2 A (lowest {r.lowest_rmsd_A} A over {r.n_modes} modes): the crystal basin was not sampled, or lies outside the output energy window.')
        if r.n_modes_with_Zn_contact == 0: print(f'    no mode has an O/S within {ZN_CONTACT_A} A of the Zn; the top mode\'s closest O/S is {r.top_Zn_OS_dist_A} A away.')
        else: print(f'    Zn-contact-restrained pick: mode {int(r.zn_restrained_mode)}, {r.zn_restrained_score:.2f} kcal/mol, RMSD {r.zn_restrained_rmsd_A} A.')


,0,1,2,3,4
form,neutral,neutral,neutral,ionised,ionised
seed,7,42,2026,7,42
n_modes,20,20,20,20,20
top_score,-7.889,-7.979,-7.912,-8.262,-8.224
top_rmsd_A,9.43,9.46,9.48,7.28,7.33
top_Zn_OS_dist_A,6.61,6.61,6.64,3.69,3.73
lowest_rmsd_A,6.91,6.69,7.2,6.92,7.33
lowest_rmsd_mode,16,4,12,8,1
lowest_rmsd_mode_score,-7.431,-7.698,-7.45,-7.846,-8.224
n_modes_with_Zn_contact,0,0,1,0,0


neutral/seed 7: no mode within 2 A (lowest 6.91 A over 20 modes): the crystal basin was not sampled, or lies outside the output energy window.
    no mode has an O/S within 2.8 A of the Zn; the top mode's closest O/S is 6.61 A away.
neutral/seed 42: no mode within 2 A (lowest 6.69 A over 20 modes): the crystal basin was not sampled, or lies outside the output energy window.
    no mode has an O/S within 2.8 A of the Zn; the top mode's closest O/S is 6.61 A away.
neutral/seed 2026: no mode within 2 A (lowest 7.2 A over 20 modes): the crystal basin was not sampled, or lies outside the output energy window.
    Zn-contact-restrained pick: mode 12, -7.45 kcal/mol, RMSD 7.2 A.
ionised/seed 7: no mode within 2 A (lowest 6.92 A over 20 modes): the crystal basin was not sampled, or lies outside the output energy window.
    no mode has an O/S within 2.8 A of the Zn; the top mode's closest O/S is 3.69 A away.
ionised/seed 42: no mode within 2 A (lowest 7.33 A over 20 modes): the crystal basin w

## Rigid-conformer redocking (separates flexibility from scoring)

The ligand is docked **as the crystal conformer with all torsions frozen** (translation and rotation only). Fast, because the search space is tiny.
- **Top pose within 2 Å**: the scoring function can recognise the crystal pose, so the failure comes from the torsional search (flexibility). More exhaustiveness, a smaller box, or pre-generated conformers can help.
- **Not within 2 Å even for the perfect conformer**: Vina's score has no funnel toward the crystal pose here (consistent with its lack of an explicit Zn-coordination term). More sampling will not fix that; a zinc-aware method is needed.

In [9]:
RUN_RIGID_REDOCK = True
RIGID_SEEDS = SEEDS[:2]

def make_rigid_pdbqt(src, dst):
    """Freeze all torsions: keep the atoms, drop BRANCH records, one ROOT, TORSDOF 0 (atom order is preserved)."""
    atoms = [l for l in Path(src).read_text().splitlines() if l.startswith(('ATOM', 'HETATM'))]
    Path(dst).write_text('\n'.join(['ROOT', *atoms, 'ENDROOT', 'TORSDOF 0']) + '\n')

if not RUN_RIGID_REDOCK:
    print('Rigid redocking skipped.')
else:
    rigid_rows = []
    for form, (ref_file, _) in FORMS.items():
        if not Path(ref_file).exists(): continue
        rigid_ligand = PATHS['REDOCKING_LIGAND'] / f'1o86_LPR_crystal_{form}_rigid.pdbqt'
        make_rigid_pdbqt(ref_file, rigid_ligand)
        ref = pdbqt_heavy_coordinates(rigid_ligand)
        for seed in RIGID_SEEDS:
            pose = PATHS['REDOCKING_DOCKING'] / f'LPR_{form}_rigid_seed_{seed}.pdbqt'
            r = subprocess.run([VINA, '--receptor', str(receptor_pdbqt), '--ligand', str(rigid_ligand),
                                '--center_x', str(box['center_x']), '--center_y', str(box['center_y']), '--center_z', str(box['center_z']),
                                '--size_x', str(box['size_x']), '--size_y', str(box['size_y']), '--size_z', str(box['size_z']),
                                '--cpu', str(DOCKING_CPU), '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(NUM_MODES), '--seed', str(seed), '--out', str(pose)],
                               text=True, capture_output=True)
            if r.returncode: raise RuntimeError(r.stderr[-1500:] or r.stdout[-1500:])
            pose.with_suffix('.log').write_text(r.stdout)
            t = mode_table(pose, ref); s = summarize_modes(t)
            rigid_rows.append({'form': form, 'seed': seed, 'top_score': s['top_score'], 'top_rmsd_A': s['top_rmsd_A'],
                               'lowest_rmsd_A': s['lowest_rmsd_A'], 'lowest_rmsd_mode': s['lowest_rmsd_mode'], 'lowest_rmsd_mode_score': s['lowest_rmsd_mode_score']})
    rigid = pd.DataFrame(rigid_rows); display(rigid)
    rigid.to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_rigid.csv', index=False)
    for r in rigid.itertuples():
        if r.top_rmsd_A <= 2.0: msg = 'the rigid crystal conformer is found: scoring can recognise the pose, so FLEXIBILITY/torsional search is the problem.'
        elif r.lowest_rmsd_A <= 2.0: msg = f'a crystal-like pose exists at mode {r.lowest_rmsd_mode} but is not ranked first: a SCORING/ranking problem.'
        else: msg = 'even the perfect rigid conformer is not placed correctly: the SCORING function has no funnel toward the crystal pose (Zn/ionic interactions); more sampling will not fix this.'
        print(f'{r.form}/seed {r.seed}: {msg}')


,form,seed,top_score,top_rmsd_A,lowest_rmsd_A,lowest_rmsd_mode,lowest_rmsd_mode_score
0,neutral,7,-14.836,0.22,0.22,1,-14.836
1,neutral,42,-14.851,0.21,0.21,1,-14.851
2,ionised,7,-14.116,0.22,0.22,1,-14.116
3,ionised,42,-14.122,0.23,0.23,1,-14.122


neutral/seed 7: the rigid crystal conformer is found: scoring can recognise the pose, so FLEXIBILITY/torsional search is the problem.
neutral/seed 42: the rigid crystal conformer is found: scoring can recognise the pose, so FLEXIBILITY/torsional search is the problem.
ionised/seed 7: the rigid crystal conformer is found: scoring can recognise the pose, so FLEXIBILITY/torsional search is the problem.
ionised/seed 42: the rigid crystal conformer is found: scoring can recognise the pose, so FLEXIBILITY/torsional search is the problem.


## Conformer-ensemble redocking (fair test of a rigid-docking protocol)

The rigid-conformer test above used the *crystal* geometry, so it only shows that the score recognises the right pose. This test does not use the crystal conformation:
1. generate conformers of lisinopril **from its SMILES** (RDKit ETKDG + MMFF),
2. dock each conformer **rigidly** (cheap),
3. refine the best rigid poses with flexible local minimisation, and rank by the Vina score.

The crystal pose is used only to measure RMSD (symmetry-aware: equivalent carboxylate oxygens and phenyl positions are matched). Redocking passes if the **top-ranked** final pose is within 2 Å. This cell runs by default (set `RUN_ENSEMBLE_REDOCK = False` to skip). Run time is roughly `ENSEMBLE_CONFORMERS` × (a few to ~15 s) on CPU; a protocol that passes this test is expensive to apply to many candidates, so it would be used for a short list.

**Replication rule:** the protocol is repeated for several seeds (each generates a different conformer set). A single success can be luck: in a first single-seed run only 2 of 100 conformers docked near the crystal pose and the winning margin over the best wrong pose was about 0.1 kcal/mol, which is well inside Vina's usual error. The summary table therefore reports, per seed, whether the top pose is within 2 Å, the best conformer RMSD, and the score margin.

In [10]:
RUN_ENSEMBLE_REDOCK = True           # set False to skip (long-running: roughly 10-15 s per conformer on 2 CPUs, unmeasured on your machine)
ENSEMBLE_CONFORMERS = 100            # first pass; use 300-1000 if the best conformer RMSD printed below is large
ENSEMBLE_RIGID_EXHAUSTIVENESS = 4
ENSEMBLE_REFINE_TOP_K = 20
ENSEMBLE_SEEDS = [7, 42, 2026]        # each seed = a different conformer set and Vina seed; passing needs MIN_PASSING_SEEDS of them

if not RUN_ENSEMBLE_REDOCK:
    print('Ensemble redocking skipped (RUN_ENSEMBLE_REDOCK = False).')
else:
    from rdkit.Chem import rdDistGeom
    from scipy.spatial.distance import cdist
    ens_dir = PATHS['REDOCKING_DOCKING'] / 'ensemble'; ens_dir.mkdir(exist_ok=True)
    ref_xyz = crystal_mol.GetConformer().GetPositions()                      # crystal heavy atoms (reference for RMSD only)
    matches = template.GetSubstructMatches(crystal_mol, uniquify=False, useChirality=False, maxMatches=20000)
    if not matches: raise RuntimeError('Could not map the SMILES-based molecule onto the crystal ligand.')

    def sym_rmsd(pose_xyz_template_order):
        """Symmetry-aware heavy-atom RMSD to the crystal pose (min over equivalent atom mappings, no superposition)."""
        return min(float(np.sqrt(np.mean(np.sum((pose_xyz_template_order[list(m)] - ref_xyz) ** 2, axis=1)))) for m in matches)

    # Self-check of the mapping logic: the crystal coordinates, written in the SMILES atom order, must give RMSD 0.
    _t = np.zeros_like(ref_xyz)
    for i, j in enumerate(matches[0]): _t[j] = ref_xyz[i]
    assert sym_rmsd(_t) < 1e-6, 'RMSD mapping self-check failed'

    def pdbqt_lines(path, first_model_only=True):
        out, started = [], False
        for l in Path(path).read_text(errors='replace').splitlines():
            if l.startswith('MODEL'): started = True; continue
            if started and l.startswith('ENDMDL') and first_model_only: break
            if l.startswith(('ATOM', 'HETATM')): out.append(l)
        return out
    def is_heavy(l): return l[77:79].strip().upper() not in {'H', 'HD', 'HS'}
    def xyz_of(l): return [float(l[30:38]), float(l[38:46]), float(l[46:54])]
    def to_template_order(input_lines, pose_lines, conf_heavy_xyz):
        """Map PDBQT heavy atoms (Meeko order) onto the SMILES atom order via the INPUT conformer coordinates."""
        inp = np.array([xyz_of(l) for l in input_lines if is_heavy(l)]); pose = np.array([xyz_of(l) for l in pose_lines if is_heavy(l)])
        d = cdist(inp, conf_heavy_xyz); j = d.argmin(axis=1)
        if d.min(axis=1).max() > 0.05 or len(set(j)) != len(j): raise RuntimeError('Atom mapping between Meeko PDBQT and RDKit conformer failed.')
        out = np.empty_like(pose); out[j] = pose
        return out
    def vina_score_text(s): return float(re.findall(r'Estimated Free Energy of Binding\s*:\s*(-?\d+\.?\d*)', s)[-1])
    def box_args_padded(pad=0.0):
        return ['--center_x', str(box['center_x']), '--center_y', str(box['center_y']), '--center_z', str(box['center_z']),
                '--size_x', str(box['size_x'] + pad), '--size_y', str(box['size_y'] + pad), '--size_z', str(box['size_z'] + pad)]
    def run_vina(ligand, extra, pad=0.0):
        r = subprocess.run([VINA, '--receptor', str(receptor_pdbqt), '--ligand', str(ligand), *box_args_padded(pad), '--cpu', str(DOCKING_CPU), *extra], text=True, capture_output=True)
        if r.returncode: raise RuntimeError(r.stderr[-1500:] or r.stdout[-1500:])
        return r.stdout

    n_heavy = template.GetNumHeavyAtoms()

    def run_ensemble(seed):
        # 1. conformers from the SMILES (no crystal coordinates are used)
        mol_h = Chem.AddHs(template)
        params = rdDistGeom.ETKDGv3(); params.randomSeed = seed; params.pruneRmsThresh = 0.5; params.numThreads = DOCKING_CPU
        cids = list(rdDistGeom.EmbedMultipleConfs(mol_h, ENSEMBLE_CONFORMERS, params))
        AllChem.MMFFOptimizeMoleculeConfs(mol_h, maxIters=1000, numThreads=DOCKING_CPU)
        # 2. rigid docking of every conformer
        records, store = [], {}      # store keeps per-conformer objects out of the DataFrame
        for cid in cids:
            tag = f's{seed}_conf_{cid:04d}'
            sdf, flex = ens_dir / f'{tag}.sdf', ens_dir / f'{tag}.pdbqt'
            sdf.write_text(Chem.MolToMolBlock(mol_h, confId=cid))
            r = subprocess.run([MEEKO_LIGAND, '-i', str(sdf), '-o', str(flex)], text=True, capture_output=True)
            if r.returncode or not flex.exists(): continue
            flex_text = flex.read_text().splitlines(); flex_atoms = pdbqt_lines(flex, first_model_only=False)
            rigid = ens_dir / f'{tag}_rigid.pdbqt'
            rigid.write_text('\n'.join([l for l in flex_text if l.startswith('REMARK')] + ['ROOT', *flex_atoms, 'ENDROOT', 'TORSDOF 0']) + '\n')
            out = ens_dir / f'{tag}_rigid_docked.pdbqt'
            run_vina(rigid, ['--exhaustiveness', str(ENSEMBLE_RIGID_EXHAUSTIVENESS), '--num_modes', '3', '--seed', str(seed), '--out', str(out)])
            conf_heavy = mol_h.GetConformer(cid).GetPositions()[:n_heavy]
            pose_lines = pdbqt_lines(out)
            score = next(float(x.split()[3]) for x in out.read_text().splitlines() if x.startswith('REMARK VINA RESULT'))
            store[cid] = {'flex': flex, 'pose': pose_lines, 'atoms': flex_atoms, 'conf_heavy': conf_heavy}
            records.append({'seed': seed, 'conformer': cid, 'rigid_score': score, 'rigid_rmsd_A': sym_rmsd(to_template_order(flex_atoms, pose_lines, conf_heavy))})
        ens = pd.DataFrame(records).sort_values('rigid_score').reset_index(drop=True)
        # 3. flexible local refinement of the best rigid poses, then final ranking by the Vina score
        final = []
        for row in ens.head(ENSEMBLE_REFINE_TOP_K).itertuples():
            o = store[row.conformer]; start_lines = []
            pose_iter = iter(o['pose'])
            for l in Path(o['flex']).read_text().splitlines():
                if l.startswith(('ATOM', 'HETATM')):
                    p = next(pose_iter); l = l[:30] + p[30:54] + l[54:]          # same atoms, rigid-pose coordinates
                start_lines.append(l)
            tag = f's{seed}_conf_{row.conformer:04d}'
            start = ens_dir / f'{tag}_start.pdbqt'; start.write_text('\n'.join(start_lines) + '\n')
            refined = ens_dir / f'{tag}_refined.pdbqt'
            try:
                # larger box for local refinement: a rigid pose near the box edge may have atoms outside the docking box
                txt = run_vina(start, ['--local_only', '--out', str(refined)], pad=12.0)
            except RuntimeError as e:
                print(f'seed {seed} conformer {row.conformer}: refinement failed ({str(e)[:80]!r}); skipped'); continue
            final.append({'seed': seed, 'conformer': row.conformer, 'rigid_score': row.rigid_score, 'rigid_rmsd_A': round(row.rigid_rmsd_A, 2),
                          'refined_score': vina_score_text(txt),
                          'refined_rmsd_A': round(sym_rmsd(to_template_order(o['atoms'], pdbqt_lines(refined), o['conf_heavy'])), 2)})
        if not final: raise RuntimeError('No pose could be refined; inspect the ensemble folder.')
        final = pd.DataFrame(final).sort_values('refined_score').reset_index(drop=True)
        final.insert(0, 'final_rank', range(1, len(final) + 1))
        return ens, final, len(cids)

    all_ens, all_final, summary_rows = [], [], []
    for seed in ENSEMBLE_SEEDS:
        ens, final, n_confs = run_ensemble(seed)
        all_ens.append(ens); all_final.append(final)
        top = final.iloc[0]
        good, wrong = final[final.refined_rmsd_A <= 2.0], final[final.refined_rmsd_A > 2.0]
        # margin > 0: the best crystal-like pose beats the best wrong pose by that many kcal/mol (differences below ~1 kcal/mol are within Vina's usual error)
        margin = (wrong.refined_score.min() - good.refined_score.min()) if len(good) and len(wrong) else np.nan
        summary_rows.append({'seed': seed, 'conformers': n_confs, 'conformers_rigid_within_2A': int((ens.rigid_rmsd_A <= 2).sum()),
                             'best_rigid_rmsd_A': round(ens.rigid_rmsd_A.min(), 2), 'top_rmsd_A': top.refined_rmsd_A, 'top_score': round(top.refined_score, 3),
                             'best_crystal_like_rank': int(good.final_rank.min()) if len(good) else np.nan,
                             'margin_vs_best_wrong_kcal': round(margin, 2) if margin == margin else np.nan, 'top_pass_2A': bool(top.refined_rmsd_A <= 2.0)})
        print(f'seed {seed}: {n_confs} conformers; top-ranked refined pose {top.refined_rmsd_A} A (score {top.refined_score:.2f})')
    pd.concat(all_ens).to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_ensemble_all_conformers_seeds.csv', index=False)
    pd.concat(all_final).to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_ensemble_final_seeds.csv', index=False)
    summary = pd.DataFrame(summary_rows); display(summary)
    summary.to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_ensemble_summary.csv', index=False)
    required = min(MIN_PASSING_SEEDS, len(ENSEMBLE_SEEDS)); n_pass = int(summary.top_pass_2A.sum())
    if n_pass >= required:
        (PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_validated_settings.json').write_text(json.dumps(
            {'protocol': 'conformer_ensemble_rigid_docking_then_flexible_refinement', 'conformers_per_seed': ENSEMBLE_CONFORMERS,
             'rigid_exhaustiveness': ENSEMBLE_RIGID_EXHAUSTIVENESS, 'refine_top_k': ENSEMBLE_REFINE_TOP_K, 'seeds': ENSEMBLE_SEEDS,
             'seeds_passing': summary[summary.top_pass_2A].seed.tolist(), 'median_margin_kcal': float(summary.margin_vs_best_wrong_kcal.median())}, indent=2))
        print(f'PASS: the top-ranked pose is within 2 A in {n_pass}/{len(ENSEMBLE_SEEDS)} seeds (required {required}). Candidates must be docked with exactly this protocol.')
        if summary.margin_vs_best_wrong_kcal.median() < 1.0: print('CAUTION: the median score margin over the best wrong pose is below ~1 kcal/mol, so the ranking is not decisive; treat the protocol as weakly validated.')
    else:
        print(f'FAIL: top-ranked pose within 2 A in only {n_pass}/{len(ENSEMBLE_SEEDS)} seeds (required {required}). See best_rigid_rmsd_A: if it is large the conformer set missed the bound geometry (try more conformers); if small but the top pose is wrong, the ranking is the problem.')
    print('\nFiles to share for review (folder: ' + str(PATHS['REDOCKING_ANALYSIS']) + '):')
    for name in ('1o86_redocking_ensemble_summary.csv', '1o86_redocking_ensemble_final_seeds.csv', '1o86_redocking_ensemble_all_conformers_seeds.csv'): print('  ', name)


seed 7: 100 conformers; top-ranked refined pose 1.3 A (score -6.96)
seed 42: 98 conformers; top-ranked refined pose 8.8 A (score -6.68)
seed 2026: 99 conformers; top-ranked refined pose 0.93 A (score -7.46)


,seed,conformers,conformers_rigid_within_2A,best_rigid_rmsd_A,top_rmsd_A,top_score,best_crystal_like_rank,margin_vs_best_wrong_kcal,top_pass_2A
0,7,100,2,1.02,1.30,-6.959,1.0,0.09,True
1,42,98,0,2.79,8.80,-6.680,NaN,NaN,False
2,2026,99,2,1.14,0.93,-7.456,1.0,0.62,True


PASS: the top-ranked pose is within 2 A in 2/3 seeds (required 2). Candidates must be docked with exactly this protocol.
CAUTION: the median score margin over the best wrong pose is below ~1 kcal/mol, so the ranking is not decisive; treat the protocol as weakly validated.

Files to share for review (folder: /content/drive/MyDrive/ACE_Workflow/redocking_1o86/analysis):
   1o86_redocking_ensemble_summary.csv
   1o86_redocking_ensemble_final_seeds.csv
   1o86_redocking_ensemble_all_conformers_seeds.csv


## High-exhaustiveness sampling study (optional, long-running)

Re-runs redocking with a much deeper search and a wide output window (`--energy_range`, many modes), so a Zn-binding pose that scores a few kcal/mol worse than the top mode can still appear. Also reports each ligand form's locally minimised crystal score. Run time scales roughly with `exhaustiveness`. Set `RUN_SAMPLING_STUDY = True` to execute.

Verdicts: **top-ranked pose within 2 Å** = validated without any restraint. **Only the Zn-contact-restrained pick within 2 Å** = validated only with that restraint (which must then also be applied to candidates). **Wrong pose scores better than the crystal minimum** = scoring-function limit. **Otherwise** = search still incomplete.

In [11]:
RUN_SAMPLING_STUDY = False          # set True to run (long)
STUDY_EXHAUSTIVENESS = 64
STUDY_SEEDS = [7]
STUDY_FORMS = ['neutral', 'ionised']
STUDY_NUM_MODES, STUDY_ENERGY_RANGE = 50, 8

if not RUN_SAMPLING_STUDY:
    print('Sampling study skipped (RUN_SAMPLING_STUDY = False).')
else:
    def study_vina(ligand, extra):
        r = subprocess.run([VINA, '--receptor', str(receptor_pdbqt),
                            '--ligand', str(ligand), '--center_x', str(box['center_x']), '--center_y', str(box['center_y']), '--center_z', str(box['center_z']),
                            '--size_x', str(box['size_x']), '--size_y', str(box['size_y']), '--size_z', str(box['size_z']), '--cpu', str(DOCKING_CPU), *extra],
                           text=True, capture_output=True)
        if r.returncode: raise RuntimeError(r.stderr[-1500:] or r.stdout[-1500:])
        return r.stdout
    study_rows, study_modes = [], []
    for form in STUDY_FORMS:
        ref_file, _ = FORMS[form]
        if not Path(ref_file).exists(): print(f'Skipping {form}: {ref_file} not found (run the diagnostics cell first).'); continue
        ref = pdbqt_heavy_coordinates(ref_file)
        lo = study_vina(ref_file, ['--local_only', '--out', str(PATHS['REDOCKING_DOCKING'] / f'LPR_{form}_crystal_local_only.pdbqt')])
        crystal_min = float(re.findall(r'Estimated Free Energy of Binding\s*:\s*(-?\d+\.?\d*)', lo)[-1])
        for seed in STUDY_SEEDS:
            pose = PATHS['REDOCKING_DOCKING'] / f'LPR_{form}_ex{STUDY_EXHAUSTIVENESS}_seed_{seed}.pdbqt'
            log = study_vina(ref_file, ['--exhaustiveness', str(STUDY_EXHAUSTIVENESS), '--num_modes', str(STUDY_NUM_MODES),
                                        '--energy_range', str(STUDY_ENERGY_RANGE), '--seed', str(seed), '--out', str(pose)])
            pose.with_suffix('.log').write_text(log)
            t = mode_table(pose, ref); study_modes.append(t.assign(form=form, seed=seed))
            s = summarize_modes(t)
            if s['top_rmsd_A'] <= 2.0: verdict = 'PASS without restraint: top-ranked pose reproduces the crystal pose'
            elif s['zn_restrained_rmsd_A'] <= 2.0: verdict = 'PASS only with the Zn-contact restraint (top-ranked pose without it is wrong)'
            elif s['top_score'] < crystal_min - 0.3: verdict = 'Vina scores a wrong pose better than the crystal minimum (scoring-function limit)'
            else: verdict = 'search still incomplete (best pose scores no better than the crystal minimum)'
            study_rows.append({'form': form, 'seed': seed, 'exhaustiveness': STUDY_EXHAUSTIVENESS, 'crystal_local_min_score': crystal_min, **s, 'verdict': verdict})
    study = pd.DataFrame(study_rows); display(study.T)
    study.to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_sampling_study.csv', index=False)
    pd.concat(study_modes).to_csv(PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_sampling_study_modes.csv', index=False)
    plain, restrained = study[study.top_rmsd_A <= 2.0], study[study.zn_restrained_rmsd_A <= 2.0]
    for rule, hit in (('unrestrained_top_pose', plain), ('zn_contact_restrained_pose', restrained)):
        if len(hit):
            (PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_validated_settings.json').write_text(json.dumps(
                {'selection_rule': rule, 'zn_contact_A': ZN_CONTACT_A if 'zn' in rule else None, 'ligand_form': hit.iloc[0].form,
                 'exhaustiveness': STUDY_EXHAUSTIVENESS, 'num_modes': STUDY_NUM_MODES, 'energy_range': STUDY_ENERGY_RANGE,
                 'seeds_passing': hit.seed.tolist(), 'seeds_run': STUDY_SEEDS}, indent=2))
            print(f'Validated setting saved (rule: {rule}). Candidate docking must use the same ligand form, search settings and pose-selection rule.')
            break


Sampling study skipped (RUN_SAMPLING_STUDY = False).
